# Kenya NLP – Drought & Flood Situation Reports
**AquaML – Pillar 4**

Classify severity and extract themes from short hydrological / drought / flood situation texts linked to Kenyan counties.

Because open, labelled county report corpora are sparse, this notebook:
1. Builds a **Kenya-specific synthetic report corpus** (realistic templates + real county names)
2. Trains severity classification (TF-IDF + linear / tree models)
3. Runs topic-style exploration
4. Shows how to plug in real sources later (NDMA bulletins, Kenya Red Cross, county water reports)

You can later replace the synthetic texts with scraped or provided real reports without changing the modelling pipeline.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import LatentDirichletAllocation

from src.evaluation import classification_metrics
from src.visualization import plot_confusion_matrix

plt.rcParams["figure.dpi"] = 120
np.random.seed(42)


## 1. Build Kenya-focused situation report corpus

In [ ]:
counties = [
    "Turkana", "Marsabit", "Wajir", "Garissa", "Isiolo", "Baringo", "West Pokot",
    "Kitui", "Makueni", "Kilifi", "Kwale", "Tana River", "Kakamega", "Vihiga",
    "Nandi", "Kajiado", "Machakos", "Taita Taveta"
]

templates = {
    "severe_flood": [
        "Severe flooding reported in {c} County. Several villages along the river inundated after heavy rainfall. Displacement of households ongoing.",
        "Flash floods in {c} have damaged water infrastructure and roads. Emergency water trucking requested by county disaster unit.",
        "River levels in {c} exceeded critical threshold. Schools closed in low-lying wards. Red Cross supporting affected families.",
    ],
    "moderate_flood": [
        "Elevated river stages observed in {c}. Monitoring continues; no widespread displacement yet.",
        "Seasonal flooding in parts of {c} within expected range. County water office advising communities near river banks.",
        "Moderate high flows in {c} after recent rains. Some boreholes temporarily inaccessible.",
    ],
    "normal": [
        "Normal hydrological conditions reported for {c} County this week. Water points functioning as expected.",
        "Routine monitoring in {c}: streamflow near long-term median. No alerts issued.",
        "Stable water supply situation in {c}. County continuing routine maintenance of boreholes and springs.",
    ],
    "drought": [
        "Prolonged dry conditions in {c}. Many shallow wells and seasonal rivers dry. Water trucking scaled up.",
        "Drought alert for {c} County. Livestock migration increasing; borehole yields declining in several wards.",
        "Critical water scarcity in {c}. Pastoral communities moving toward remaining permanent water sources.",
    ],
    "water_quality": [
        "Elevated salinity reported at several boreholes in {c}. Communities advised to use alternative sources where available.",
        "Water quality sampling in {c} shows high EC in a subset of ASAL boreholes. Treatment options under review.",
        "Fluoride concerns flagged for a cluster of wells in {c}. County coordinating with national lab for confirmation.",
    ],
}

rows = []
for i in range(500):
    sev = np.random.choice(list(templates.keys()), p=[0.18, 0.22, 0.25, 0.22, 0.13])
    c = np.random.choice(counties)
    text = np.random.choice(templates[sev]).format(c=c)
    if np.random.rand() > 0.5:
        text += " Local authorities coordinating with national agencies."
    rows.append({
        "report_id": f"KE{i:04d}",
        "county": c,
        "text": text,
        "severity": sev,
    })

reports = pd.DataFrame(rows)
print(reports["severity"].value_counts())
print(reports.sample(3, random_state=1)[["county", "severity", "text"]].to_string())
reports.to_csv(ROOT / "data/kenya/kenya_situation_reports_synthetic.csv", index=False)
print("\nSaved synthetic corpus to data/kenya/kenya_situation_reports_synthetic.csv")


## 2. Severity classification

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(reports["severity"])
class_names = list(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    reports["text"], y, test_size=0.25, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1, 2), stop_words="english", min_df=2)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf.fit(X_tr, y_train)
pred = clf.predict(X_te)

print(classification_metrics(y_test, pred))
print(classification_report(y_test, pred, target_names=class_names, digits=3))


In [ ]:
cm = confusion_matrix(y_test, pred)
plot_confusion_matrix(cm, class_names, title="Kenya situation report severity")
plt.show()


## 3. Most indicative terms per class

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
for i, cls in enumerate(class_names):
    coefs = clf.coef_[i]
    top = feature_names[np.argsort(coefs)[-8:][::-1]]
    print(f"{cls:16s}: {', '.join(top)}")


## 4. Topic exploration (LDA)

In [ ]:
X_all = tfidf.fit_transform(reports["text"])
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=25)
lda.fit(X_all)
print("Top words per topic:")
for topic_idx, topic in enumerate(lda.components_):
    top_words = feature_names[np.argsort(topic)[-10:][::-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")


## 5. Linking to real Kenya sources (next step)

Replace the synthetic corpus with real text from:
- **NDMA** (National Drought Management Authority) early-warning bulletins
- County disaster / water office situation reports
- Kenya Red Cross / UN OCHA flash updates
- Local radio / newspaper summaries of flood and drought events

Keep the same columns (`text`, `severity` or event type, `county`, `date`) and the modelling cells above remain valid.

You can also join report counties to the water-point and borehole tables for multi-modal analysis.


## Takeaways – NLP Pillar

- Domain language around Kenyan floods, drought and water quality is highly separable with classic TF-IDF models.
- Interpretable coefficients help water managers trust the system.
- Synthetic data is only a scaffold; the pipeline is ready for real NDMA / county reports.

**Outputs**
- `data/kenya/kenya_situation_reports_synthetic.csv`
